In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

from xgboost import XGBRegressor
import optuna

SEED = 42
np.random.seed(SEED)


In [2]:
train = pd.read_csv('/kaggle/input/first-competition-exhibition/train.csv')
test  = pd.read_csv('/kaggle/input/first-competition-exhibition/test.csv')

test_ids = test['id']

train = train.drop(columns=['id', 'Row#'])
test  = test.drop(columns=['id', 'Row#'])


In [3]:
def create_features(df, kmeans=None, scaler=None, fit=False):
    data = df.copy()

    # ---- Bees ----
    data['total_bees'] = (
        data['honeybee'] +
        data['bumbles'] +
        data['andrena'] +
        data['osmia']
    )

    data['bees_per_clone'] = data['total_bees'] / (data['clonesize'] + 1e-6)
    data['osmia_honeybee_inter'] = data['osmia'] * data['honeybee']

    # ---- Temperature ----
    data['temp_range'] = (
        data['MaxOfUpperTRange'] -
        data['MinOfLowerTRange']
    )

    data['avg_temp'] = (
        data['AverageOfUpperTRange'] +
        data['AverageOfLowerTRange']
    ) / 2

    data['temp_x_rain'] = data['avg_temp'] * data['RainingDays']

    # ---- Non-linear ----
    for col in ['clonesize', 'total_bees', 'fruitmass', 'seeds']:
        data[f'log_{col}'] = np.log1p(data[col])

    data['fruit_seed_ratio'] = data['fruitmass'] / (data['seeds'] + 1e-6)

    # ---- Clustering ----
    cluster_cols = ['clonesize', 'total_bees', 'avg_temp', 'RainingDays']

    if fit:
        scaler = StandardScaler()
        scaled = scaler.fit_transform(data[cluster_cols])

        kmeans = KMeans(
            n_clusters=6,
            random_state=SEED,
            n_init=20
        )
        data['cluster'] = kmeans.fit_predict(scaled)
    else:
        scaled = scaler.transform(data[cluster_cols])
        data['cluster'] = kmeans.predict(scaled)

    return data, kmeans, scaler

train_fe, kmeans, scaler = create_features(train, fit=True)
test_fe, _, _ = create_features(test, kmeans=kmeans, scaler=scaler)

In [4]:
def add_target_encoding(train_df, test_df, target_col, group_col, n_splits=5):
    train_df = train_df.copy()
    test_df  = test_df.copy()

    train_df[f'{group_col}_te'] = 0.0

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)

    for tr_idx, val_idx in kf.split(train_df):
        tr, val = train_df.iloc[tr_idx], train_df.iloc[val_idx]
        stats = tr.groupby(group_col)[target_col].median()
        train_df.iloc[val_idx, train_df.columns.get_loc(f'{group_col}_te')] = \
            val[group_col].map(stats)

    global_median = train_df[target_col].median()
    train_df[f'{group_col}_te'].fillna(global_median, inplace=True)

    test_stats = train_df.groupby(group_col)[target_col].median()
    test_df[f'{group_col}_te'] = test_df[group_col].map(test_stats)
    test_df[f'{group_col}_te'].fillna(global_median, inplace=True)

    return train_df, test_df

train_fe, test_fe = add_target_encoding(
    train_fe, test_fe,
    target_col='yield',
    group_col='cluster'
)


In [5]:
X = train_fe.drop(columns=['yield'])
y = train_fe['yield']

y_min, y_max = y.min(), y.max()


In [6]:
def objective(trial, alpha):
    params = {
        'n_estimators': 5000,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.05),
        'max_depth': trial.suggest_int('max_depth', 4, 6),
        'min_child_weight': trial.suggest_int('min_child_weight', 3, 8),
        'subsample': trial.suggest_float('subsample', 0.7, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 0.9),
        'gamma': trial.suggest_float('gamma', 0.3, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.3, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 1.0, 3.0),
        'objective': 'reg:quantileerror',
        'quantile_alpha': alpha,
        'tree_method': 'gpu_hist',
        'random_state': SEED
    }

    kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
    maes = []

    for tr_idx, val_idx in kf.split(X):
        model = XGBRegressor(**params)
        model.fit(
            X.iloc[tr_idx], y.iloc[tr_idx],
            eval_set=[(X.iloc[val_idx], y.iloc[val_idx])],
            early_stopping_rounds=200,
            verbose=False
        )
        preds = model.predict(X.iloc[val_idx])
        maes.append(mean_absolute_error(y.iloc[val_idx], preds))

    return np.mean(maes)

In [7]:
quantiles = [0.3, 0.5, 0.7]
models = []
test_preds = []

best_params_store = {}

for alpha in quantiles:
    study = optuna.create_study(direction='minimize')
    study.optimize(lambda t: objective(t, alpha), n_trials=40)
    best_params = study.best_params

    model = XGBRegressor(
        **best_params,
        n_estimators=5000,
        objective='reg:quantileerror',
        quantile_alpha=alpha,
        tree_method='gpu_hist',
        random_state=SEED
    )

    model.fit(X, y, verbose=False)
    models.append(model)
    test_preds.append(model.predict(test_fe))

[I 2025-12-26 07:35:04,266] A new study created in memory with name: no-name-db03cf36-00b4-4fee-8bad-6f14e06f35ca
[I 2025-12-26 07:35:17,930] Trial 0 finished with value: 270.84394134879165 and parameters: {'learning_rate': 0.029030838209519474, 'max_depth': 4, 'min_child_weight': 5, 'subsample': 0.806168117073234, 'colsample_bytree': 0.7838638831442752, 'gamma': 0.39596368400284676, 'reg_alpha': 0.8121398441236716, 'reg_lambda': 1.9935707719737448}. Best is trial 0 with value: 270.84394134879165.
[I 2025-12-26 07:35:29,215] Trial 1 finished with value: 270.833102881901 and parameters: {'learning_rate': 0.047177618548147386, 'max_depth': 5, 'min_child_weight': 4, 'subsample': 0.7613242137386428, 'colsample_bytree': 0.7876223453313966, 'gamma': 0.5849590483942227, 'reg_alpha': 0.5849226230735487, 'reg_lambda': 1.1171032099057823}. Best is trial 1 with value: 270.833102881901.
[I 2025-12-26 07:35:54,557] Trial 2 finished with value: 271.5161867359062 and parameters: {'learning_rate': 0.0

In [8]:
final_test_preds = (
    0.2 * test_preds[0] +
    0.6 * test_preds[1] +
    0.2 * test_preds[2]
)

final_test_preds = np.clip(final_test_preds, y_min, y_max)

submission = pd.DataFrame({
    'id': test_ids,
    'yield': final_test_preds
})

submission.to_csv('submission.csv', index=False)
submission.head()

,id,yield
0,15000,7492.945801
1,15001,5841.752930
2,15002,6541.151855
3,15003,4636.203613
4,15004,5899.436523
